In [ ]:
import torch
from simple_neural_mpc.robots import Unicycle
from simple_neural_mpc.neural_modeling.dataset import UnicycleDataset
from simple_neural_mpc.neural_modeling.dataset.datamodule import Datamodule
from simple_neural_mpc.utils import project_root
import numpy as np
from simple_neural_mpc.config.neural_config import DatasetConfig
from simple_neural_mpc.config.neural_config import TrainerConfig
np.set_printoptions(precision=3, suppress=True)

seed = 134
torch.random.manual_seed(seed)
np.random.seed(seed)
      
robot = Unicycle()
if DatasetConfig.name == 'derivative':
    dataset = None if DatasetConfig.load_data else UnicycleDataset.generate_derivative_data(robot)
elif DatasetConfig.name == 'state':
    dataset = None if DatasetConfig.load_data else UnicycleDataset.generate_trajectory_data(robot)
else:
    raise ValueError(f"Unknown dataset name: {DatasetConfig.name}")

TrainerConfig.wandb_project = f"{TrainerConfig.wandb_project}_{DatasetConfig.name}"
datamodule = Datamodule(dataset, savedpath=f'{project_root()}/data/{DatasetConfig.name}')

In [ ]:
from simple_neural_mpc.neural_modeling.learner.next_state_learner import (
    NextStateLearner,
)
from simple_neural_mpc.neural_modeling.learner.derivative_learner import (
    DerivativeLearner,
)
from simple_neural_mpc.neural_modeling.learner.trainer import Trainer
from simple_neural_mpc.config.neural_config import TrainerConfig

if DatasetConfig.name == "derivative":
    learner = DerivativeLearner(3, 2, in_mpc=False)
elif DatasetConfig.name == "state":
    learner = NextStateLearner(3, 2, is_pinn=True, in_mpc=False)
trainer = Trainer()
trainer.fit(learner, datamodule)

In [ ]:
import torch

learner.load_state_dict(torch.load(f'{TrainerConfig.ckpt_path}/unicycle.pth'))
test_data = UnicycleDataset.generate_test_data(robot)

learner.test_traj(test_data)